In [1]:
%%writefile main.py
"""
Orbit Wars - "The Grand Strategist v2.0"
Incorporates:
- 1.1x Fleet Speed Over-Commitment (Speed boosting & auto-garrison).
- 5-Iteration Orbital Intercept Solver.
- 4P Kingmaker Dynamics (Target leader, ignore underdogs).
- Bucket-Brigade Logistics (Short-hop banking).
- Anti-Fray Protocols (No cluster busting without a fortified core).
"""

import math
from collections import defaultdict, namedtuple

# ============================================================
# Core Configuration & Tactics
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5
HORIZON = 100
TOTAL_STEPS = 500

# Micro-Tactics
OVERCOMMIT_RATIO = 1.1        # 10% extra ships for Speed Boost & Garrison
MAX_FLIGHT_TURNS = 35         # Hard cap on long-distance snipes
CORE_FORTIFIED_THRESH = 20    # Average ships needed in core to allow Cluster Busting

# Multipliers
KINGMAKER_BONUS = 1.5         # Bonus for attacking the current production leader
NON_LEADER_PENALTY = 0.8      # Penalty for attacking the underdog
FLANK_EXPANSION_BONUS = 1.45
PINCER_MANEUVER_BONUS = 1.35

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet = namedtuple("Fleet",["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

# ============================================================
# Physics & 5-Iteration Solver
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    ratio = max(0.0, min(1.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

class PlanetTracker:
    def __init__(self, planets, initial_planets, ang_vel, comets, comet_ids):
        self.pos = {}
        for p in planets:
            self.pos[p.id] =[self._calc(p, t, ang_vel, initial_planets, comets, comet_ids) for t in range(HORIZON + 1)]
            
    def _calc(self, p, t, ang_vel, initial_planets, comets, comet_ids):
        if p.id in comet_ids:
            for c in comets:
                if p.id in c.get("planet_ids", []):
                    idx = c["planet_ids"].index(p.id)
                    f_idx = c.get("path_index", 0) + t
                    if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]):
                        return c["paths"][idx][f_idx]
            return None
        init = initial_planets.get(p.id)
        if not init: return (p.x, p.y)
        r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
        if r + init.radius >= 50.0: return (p.x, p.y)
        ang = math.atan2(p.y - CENTER_Y, p.x - CENTER_X) + ang_vel * t
        return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight_iterative(src, tgt, ships, tracker):
    """
    5-Iteration Root-Finding Solver.
    Perfectly synchronizes the logarithmic speed of the fleet with the orbital velocity of the target.
    """
    speed = fleet_speed(ships)
    sx, sy, sr = src.x, src.y, src.radius
    tr = tgt.radius
    
    # Initial guess: Target's current position
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    
    for _ in range(5):
        dist = math.hypot(tx - sx, ty - sy)
        flight_dist = max(0.0, dist - sr - 0.1 - tr)
        eta = flight_dist / speed
        
        turn_idx = int(math.ceil(eta))
        if turn_idx >= HORIZON: return None, 999
        pos = tracker.pos[tgt.id][turn_idx]
        if not pos: return None, 999  # Comet disappeared
        tx, ty = pos

    angle = math.atan2(ty - sy, tx - sx)
    turn_idx = int(math.ceil(eta))
    
    # Verify Sun Avoidance on the exact final trajectory
    start_x = sx + math.cos(angle) * (sr + 0.1)
    start_y = sy + math.sin(angle) * (sr + 0.1)
    end_x = start_x + math.cos(angle) * flight_dist
    end_y = start_y + math.sin(angle) * flight_dist
    if point_to_segment_dist(CENTER_X, CENTER_Y, start_x, start_y, end_x, end_y) <= SUN_R + SUN_SAFETY:
        return None, 999
        
    return angle, turn_idx

# ============================================================
# Logistics & Timeline Simulation
# ============================================================
def simulate_timeline(pid, planet, active_fleets, player):
    by_turn = defaultdict(int)
    for f in active_fleets:
        if f['target'] == pid:
            by_turn[f['eta']] += f['ships'] if f['owner'] == player else -f['ships']
            
    curr_owner = planet.owner
    curr_ships = float(planet.ships)
    min_owned_ships = curr_ships if curr_owner == player else 0
    fall_turn = None
    
    for t in range(1, 40):
        if curr_owner != -1: curr_ships += planet.production
        if t in by_turn:
            curr_ships += by_turn[t]
            if curr_ships < 0:
                curr_owner = -1 if curr_owner == player else player
                curr_ships = -curr_ships
                if fall_turn is None and curr_owner != player: fall_turn = t
        if curr_owner == player:
            min_owned_ships = min(min_owned_ships, curr_ships)
            
    return curr_owner, curr_ships, fall_turn, max(0, int(min_owned_ships))

def get_sector(planet, core_angle):
    angle = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    diff = (angle - core_angle + math.pi) % (2 * math.pi) - math.pi
    if abs(diff) <= 0.78: return 'core'
    elif 0.78 < diff < 2.35: return 'right_flank'
    elif -2.35 < diff < -0.78: return 'left_flank'
    else: return 'enemy_core'

# ============================================================
# Main Agent Engine
# ============================================================
def agent(obs, config=None):
    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    planets =[Planet(*p) for p in (obs.get("planets", []) if is_dict else getattr(obs, "planets", []))]
    fleets =[Fleet(*f) for f in (obs.get("fleets", []) if is_dict else getattr(obs, "fleets",[]))]
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    raw_init = obs.get("initial_planets", []) if is_dict else getattr(obs, "initial_planets",[])
    comets = obs.get("comets",[]) if is_dict else getattr(obs, "comets", [])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids",[]))
    
    tracker = PlanetTracker(planets, {Planet(*p).id: Planet(*p) for p in raw_init}, ang_vel, comets, comet_ids)
    
    # 4P Kingmaker Dynamics (Identify the Leader)
    player_prod = defaultdict(int)
    for p in planets:
        if p.owner != -1: player_prod[p.owner] += p.production
    prod_leader = max(player_prod, key=player_prod.get) if player_prod else -1
    is_4p = len(player_prod) > 2
    
    my_planets =[p for p in planets if p.owner == player]
    enemy_planets =[p for p in planets if p.owner not in (-1, player)]
    if not my_planets: return[]

    # Map Control & Sectoring
    capital = max(my_planets, key=lambda p: p.ships + (p.production * 10))
    core_angle = math.atan2(capital.y - CENTER_Y, capital.x - CENTER_X)
    
    core_planets =[p for p in my_planets if get_sector(p, core_angle) == 'core']
    is_core_fortified = sum(p.ships for p in core_planets) > (len(core_planets) * CORE_FORTIFIED_THRESH)

    moves = []
    budget = {}
    dist_to_front = {p.id: min([math.hypot(p.x - e.x, p.y - e.y) for e in enemy_planets]) if enemy_planets else 999 for p in my_planets}
    
    # Active Flights Ledger
    active_flights =[]
    for f in fleets:
        # Simple tracker for existing fleets
        for N in range(1, HORIZON):
            pos = tracker.pos[f.from_planet_id][N] if f.from_planet_id in tracker.pos else None # Approximate tgt
            if pos: break # Simplified for existing flights
        active_flights.append({'target': f.from_planet_id, 'eta': 10, 'owner': f.owner, 'ships': f.ships}) # Stubbed to save CPU

    hubs, spokes = [],[]
    for p in my_planets:
        _, _, fall_turn, min_ships = simulate_timeline(p.id, p, active_flights, player)
        budget[p.id] = min_ships
        if fall_turn is not None and min_ships <= 0:
            budget[p.id] = int(p.ships) # Evacuate
            spokes.append(p)
            continue
            
        if dist_to_front[p.id] > 35.0: spokes.append(p)
        else: hubs.append(p)
            
    if not hubs: hubs = [capital]

    # 1. Bucket-Brigade Logistics (Short-Hop Banking)
    for sp in spokes:
        avail = budget[sp.id] - 8
        if avail > 10:
            # Find the closest friendly planet that is nearer to the front than this spoke
            valid_targets = [f for f in my_planets if f.id != sp.id and dist_to_front[f.id] < dist_to_front[sp.id]]
            if valid_targets:
                best_hop = min(valid_targets, key=lambda f: math.hypot(sp.x - f.x, sp.y - f.y))
                angle, t = plan_flight_iterative(sp, best_hop, avail, tracker)
                if angle is not None:
                    moves.append([sp.id, angle, avail])
                    budget[sp.id] -= avail
                    active_flights.append({'target': best_hop.id, 'eta': t, 'owner': player, 'ships': avail})

    # 2. Strike Coordination
    roi_matrix = []
    for hub in hubs:
        avail = budget[hub.id]
        if avail < 10: continue
        hub_sector = get_sector(hub, core_angle)
        
        for tgt in planets:
            if tgt.owner == player: continue
            
            # Predict intercept using 5-iteration solver with assumed 1.1x speed boost
            assumed_ships = min(avail, int(tgt.ships * OVERCOMMIT_RATIO) + 5)
            angle, eta = plan_flight_iterative(hub, tgt, assumed_ships, tracker)
            
            # Anti-Long Distance Sniping Rule
            if not angle or eta > MAX_FLIGHT_TURNS: continue 
            
            proj_owner, proj_ships, _, _ = simulate_timeline(tgt.id, tgt, active_flights, player)
            base_cost = proj_ships + (tgt.production * eta if proj_owner != -1 else 0) + 1
            
            # THE SPEED BOOST: Over-commit by 10%
            boosted_cost = int(math.ceil(base_cost * OVERCOMMIT_RATIO))
            if boosted_cost >= avail: continue
            
            tgt_sector = get_sector(tgt, core_angle)
            map_bonus = 1.0
            if tgt_sector in ('left_flank', 'right_flank'): map_bonus = FLANK_EXPANSION_BONUS
            elif tgt_sector == 'enemy_core' and hub_sector in ('left_flank', 'right_flank'): map_bonus = PINCER_MANEUVER_BONUS
            
            # Kingmaker Multiplier
            if is_4p and tgt.owner != -1:
                if tgt.owner == prod_leader: map_bonus *= KINGMAKER_BONUS
                else: map_bonus *= NON_LEADER_PENALTY
            
            # Anti-Fray / Cluster Buster Logic
            neighbors = sum(1 for op in planets if op.id != tgt.id and math.hypot(op.x - tgt.x, op.y - tgt.y) < 22.0)
            if neighbors > 1 and tgt.owner != -1:
                if is_core_fortified: map_bonus *= 1.4 # We are secure, crack the cluster
                else: map_bonus *= 0.4 # We are weak, avoid the fray
                
            dist_penalty = 1.0 / (eta ** 0.8)
            val = (tgt.production * max(0, (TOTAL_STEPS - step - eta))) * map_bonus
            
            roi = (val / boosted_cost) * dist_penalty
            roi_matrix.append((roi, hub.id, tgt.id, boosted_cost, angle))

    # 3. Execute Strikes
    roi_matrix.sort(key=lambda x: x[0], reverse=True)
    for roi, src_id, tgt_id, cost, angle in roi_matrix:
        if budget[src_id] >= cost:
            moves.append([src_id, angle, cost])
            budget[src_id] -= cost
            active_flights.append({'target': tgt_id, 'eta': 1, 'owner': player, 'ships': cost})

    return moves

Writing main.py
